# Project 3:
Author: Chi Hang (Philip) Cheung

1. Using any of the three classifiers described in chapter 6 of Natural Language Processing with Python, and any features you can think of, build the best name gender classifier you can.
2. Begin by splitting the Names Corpus into three subsets: 500 words for the test set, 500 words for the dev-test set, and the remaining 6900 words for the training set. 
3. Then, starting with the example name gender classifier, make incremental improvements. 
4. Use the dev-test set to check your progress. Once you are satisfied with your classifier, check its final performance on the test set.
5. How does the performance on the test set compare to the performance on the dev-test set? Is this what you'd expect?


Source: Natural Language Processing with Python, exercise 6.10.2.

In [9]:
import nltk
from nltk.corpus import names
import random
from itertools import combinations

### Loading the Name dataset and combining male and female into one file:

In [10]:
name_data = [(n, 'male') for n in names.words('male.txt')] + [(n, 'female') for n in names.words('female.txt')]

print(name_data[:10])
print('-'*80)
print(f'Total length of the name list: {len(name_data)}')

[('Aamir', 'male'), ('Aaron', 'male'), ('Abbey', 'male'), ('Abbie', 'male'), ('Abbot', 'male'), ('Abbott', 'male'), ('Abby', 'male'), ('Abdel', 'male'), ('Abdul', 'male'), ('Abdulkarim', 'male')]
--------------------------------------------------------------------------------
Total length of the name list: 7944


### We will shuffle the data and divide them into test, dev, and train dataset:

In [11]:
random.seed(41)
random.shuffle(name_data)

#Splitting into train, test, dev dataset
test_set = name_data[:500]
dev_set = name_data[500:1000]
train_set = name_data[1000:]

print(f'length of test: {len(test_set)} | length of dev: {len(dev_set)} | length of train: {len(train_set)}')

length of test: 500 | length of dev: 500 | length of train: 6944


### Feature extraction experiments using Maximum Entropy Classifier

### Experiment 1:
#### Baseline = only use last letter of the name

In [12]:
def gender_feature_baseline(name):
    clean_names = name.strip().lower()
    return {'last_letter': clean_names[-1]}

feature = [(gender_feature_baseline(n), gender) for (n, gender) in train_set]

maxent_classifier = nltk.MaxentClassifier.train(feature, algorithm='IIS', trace=0)

#Evaluation:
dev_features = [(gender_feature_baseline(n), gender) for (n, gender) in dev_set]
dev_accuracy = nltk.classify.accuracy(maxent_classifier, dev_features)
print(f'Dev set MaxEnt Accuracy: {dev_accuracy}')

# Show the top 10 most informative rules the model discovered
print("\nMost Informative Features:")
maxent_classifier.show_most_informative_features(10)


Dev set MaxEnt Accuracy: 0.78

Most Informative Features:
   6.644 last_letter=='c' and label is 'male'
  -4.861 last_letter=='a' and label is 'male'
  -3.273 last_letter=='k' and label is 'female'
  -3.000 last_letter=='p' and label is 'female'
  -2.524 last_letter=='f' and label is 'female'
  -2.000 last_letter=='v' and label is 'female'
  -1.916 last_letter=='i' and label is 'male'
  -1.795 last_letter=='d' and label is 'female'
  -1.730 last_letter=='m' and label is 'female'
  -1.585 last_letter=='o' and label is 'female'


The baseline accuracy of name classification with Maximum Entropy is 0.78. We should test other features in combination with the last letter to see if the accuracy can be improved.

### Experiment 2:
#### we will setup a auto-tuning algorithm to mix and match features and tune the classifier to have the best accuracy score

In [ ]:
#Define a list of features:
feature_list = [
    'last_letter',
    'first_letter',
    'first_2_letters',
    'length',
    'end_with_vowel',
    'vowel_count'
]

#Define the dynamic function:
def dynamic_feature(name, feature_set):
    clean_name = name.strip().lower()
    features = {}
    if 'last_letter' in feature_set:
        features['last_letter'] = clean_name[-1]
    if 'first_letter' in feature_set:
        features['first_letter'] = clean_name[0]
    if 'first_2_letters' in feature_set:
        features['first_2_letters'] = clean_name[:2]
    if 'length' in feature_set:
        features['length'] = len(clean_name)
    if 'end_with_vowel' in feature_set:
        features['end_with_vowel'] = clean_name[-1] in 'aeiou'
    if 'vowel_count' in feature_set:
        features['vowel_count'] = sum(1 for char in clean_name if char in 'aeiou')
    return features

#Define max number of feature allowed per run:
#Here we will allow 1 - 3 features at a run.
max_feat = min(len(feature_list), 3)

#Loop through the MaxEnt with different combination of the features and record the best feature set
best_accuracy = 0
best_feature_set = []

for i in range(1, max_feat +1):
    for feature in itertools.combinations(feature_list, i):
        train_feature_extract = [(dynamic_feature(n, feature), g) for n, g in train_set]
        dev_feature_extract = [(dynamic_feature(n, feature), g) for n, g in dev_set]
        maxent_classifier = nltk.MaxentClassifier.train(train_feature_extract, max_iter=10)
        accuracy =  nltk.classify.accuracy(maxent_classifier, dev_feature_extract)
        print(f'Current feature set: {feature}')
        print(f'Current accuracy: {accuracy}')

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_feature_set = feature
print('-'*80)
print(f'best overall accuracy: {best_accuracy}')
print(f'best overall feature set: {best_feature_set}')


  ==> Training (10 iterations)

      Iteration    Log Likelihood    Accuracy
      ---------------------------------------
             1          -0.69315        0.371
             2          -0.37473        0.761
             3          -0.37434        0.761
             4          -0.37411        0.761
             5          -0.37396        0.761
             6          -0.37385        0.761
             7          -0.37377        0.761
             8          -0.37370        0.761
             9          -0.37365        0.761
         Final          -0.37361        0.761
Current feature set: ('last_letter',)
Current accuracy: 0.78
  ==> Training (10 iterations)

      Iteration    Log Likelihood    Accuracy
      ---------------------------------------
             1          -0.69315        0.371
             2          -0.58834        0.646
             3          -0.58834        0.646
             4          -0.58834        0.646
             5          -0.58834        0.646
 

The best accuracy is 0.826 with feature sets 'last_letter', 'first_2_letters', and 'vowel_count'. This is roughly 4% better the baseline with only the last letter.

### We will apply this model to the test_set and get the name classifications

In [20]:
best_features = [
    'last_letter',    
    'first_2_letters',    
    'vowel_count'
]

train_feature_extract = [(dynamic_feature(n, best_features), g) for n, g in train_set]
test_feature_extract = [(dynamic_feature(n, best_features), g) for n, g in test_set]
maxent_classifier = nltk.MaxentClassifier.train(train_feature_extract, max_iter=10)
accuracy =  nltk.classify.accuracy(maxent_classifier, test_feature_extract)

print(f'Accuracy against test_set: {accuracy}')

  ==> Training (10 iterations)

      Iteration    Log Likelihood    Accuracy
      ---------------------------------------
             1          -0.69315        0.371
             2          -0.45321        0.761
             3          -0.39498        0.793
             4          -0.36660        0.797
             5          -0.35040        0.802
             6          -0.34015        0.803
             7          -0.33320        0.805
             8          -0.32824        0.806
             9          -0.32456        0.806
         Final          -0.32175        0.805
Accuracy against test_set: 0.804


The accuracy of the classification against the test set is 0.804, which is slightly lower than the dev set. This is an expected outcome — the feature set was chosen by optimizing for the highest dev set accuracy across all combinations tested. As a result, the dev set accuracy is slightly inflated or more overfitted for the dev set. The test set, which was never used during model selection, provides a more generalized and realistic estimate of the classifier's true performance on unseen data.

Maximum Entropy classifier was chosen for the classification because of it offers the highest accuracy over decision tree and Navie Bayes. The former is prone to overfitted without ensemble method and the latter assumes feature independence. Although Maximum Entropy Classification is slower to train, it over comes the cons of the other two methods, offering the highest accuracy in the name classification.